# 配套实践 09-01：手工计算 Attention 与 mask

本练习使用 NumPy 完成缩放点积 Attention。我们将让两个不同查询读取同一组机器人历史，随后观察缩放、padding mask 和 causal mask 怎样改变注意力权重。依赖：NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/09-attention-and-transformer/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import numpy as np  # 计算点积、Softmax 和加权读取
import matplotlib.pyplot as plt  # 绘制注意力权重与 mask 矩阵
np.random.seed(91)  # 固定随机向量以便重复观察
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 两个查询读取同一段历史

四个历史位置分别表示看见目标、末端接近、发生接触和背景变化。Key 用来匹配查询，Value 保存实际要取回的内容。下面的人工向量只用于展示计算关系，不代表真实模型会直接使用这些手工语义。

In [ ]:
history_names = ["See target", "Approach", "Contact", "Light change"]  # 使用通用英文字体标记四个历史位置
keys = np.array([[0.9, 0.1, 0.0], [0.4, 0.7, 0.1], [0.1, 0.5, 1.0], [0.0, 0.1, 0.2]], dtype=np.float64)  # 定义四个历史位置的匹配 Key
values = np.array([[0.8, 0.1], [0.5, 0.4], [0.1, 1.0], [0.2, 0.0]], dtype=np.float64)  # 定义四个位置实际提供的 Value
queries = np.array([[0.1, 0.4, 1.0], [1.0, 0.1, 0.0]], dtype=np.float64)  # 定义接触查询和目标查询
query_names = ["Should close gripper?", "Where is the target?"]  # 为两个查询准备可读名称
def stable_softmax(scores, axis=-1):  # 定义减去最大值后的数值稳定 Softmax
    shifted = scores - np.max(scores, axis=axis, keepdims=True)  # 平移分数避免指数溢出
    exponentials = np.exp(shifted)  # 把平移后的相似度转成正数
    return exponentials / exponentials.sum(axis=axis, keepdims=True)  # 归一化得到和为一的权重
def scaled_dot_attention(query_matrix, key_matrix, value_matrix, additive_mask=None):  # 定义完整缩放点积 Attention
    scores = query_matrix @ key_matrix.T / np.sqrt(key_matrix.shape[-1])  # 计算查询与全部 Key 的缩放点积
    if additive_mask is not None:  # 判断是否需要排除某些读取位置
        scores = scores + additive_mask  # 给禁止位置加入很大的负数
    weights = stable_softmax(scores, axis=-1)  # 把匹配分数变成相对读取权重
    context = weights @ value_matrix  # 根据权重对 Value 进行加权求和
    return context, weights, scores  # 返回读取结果、权重和遮挡后的原始分数
contexts, attention_weights, attention_scores = scaled_dot_attention(queries, keys, values)  # 让两个查询读取同一组历史 Value
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)  # 创建两个查询的权重柱状图
for query_index, axis in enumerate(axes):  # 依次绘制两个查询的历史读取比例
    axis.barh(history_names, attention_weights[query_index], color="#2563eb")  # 绘制当前查询对四个历史位置的权重
    axis.set(title=query_names[query_index], xlabel="Attention weight", xlim=(0.0, 0.6))  # 使用通用英文字体标注权重含义
    for history_index, weight in enumerate(attention_weights[query_index]):  # 依次读取柱子的具体权重
        axis.text(weight + 0.01, history_index, f"{weight:.2f}", va="center")  # 在柱子末端显示两位小数权重
fig.suptitle("Different queries read different parts of the same history")  # 强调查询决定读取位置
fig.tight_layout()  # 调整子图间距避免文字重叠
plt.show()  # 显示两个查询的注意力结果

**怎样理解结果：** “是否闭合夹爪”的查询在第三个维度上较强，因此更偏向接触历史；“目标在哪里”的查询则更偏向最早的目标视觉信息。同一组 Key 和 Value 并没有固定的唯一重要位置，Query 改变后读取权重与上下文都会改变。

## 2. 为什么点积需要缩放

特征维度增大时，随机向量点积的波动通常也会增大。未经缩放的分数容易让 Softmax 很快集中到少数位置。下面在相同随机分数上比较是否除以 $\sqrt{d_k}$。

In [ ]:
feature_dimension = 64  # 设置较高的 Query 和 Key 特征维度
random_query = np.random.randn(1, feature_dimension)  # 生成一个随机查询向量
random_keys = np.random.randn(12, feature_dimension)  # 生成十二个随机 Key
raw_scores = random_query @ random_keys.T  # 计算未经缩放的点积分数
scaled_scores = raw_scores / np.sqrt(feature_dimension)  # 使用特征维度平方根缩放点积
raw_weights = stable_softmax(raw_scores, axis=-1).ravel()  # 得到未经缩放的 Softmax 权重
scaled_weights = stable_softmax(scaled_scores, axis=-1).ravel()  # 得到缩放后的 Softmax 权重
positions = np.arange(len(random_keys))  # 建立十二个 Key 的横轴位置
fig, ax = plt.subplots(figsize=(8.5, 3.5))  # 创建缩放前后的权重对比图
ax.plot(positions, raw_weights, "o-", label="Without scaling")  # 绘制未经缩放时更尖锐的权重
ax.plot(positions, scaled_weights, "s-", label="Scaled by sqrt(d_k)")  # 绘制缩放后的权重
ax.set(title="Scaling keeps Softmax from saturating too early", xlabel="Key position", ylabel="Attention weight")  # 使用通用英文字体标注缩放作用
ax.legend()  # 显示缩放与未缩放曲线图例
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示缩放对权重分布的影响

**怎样理解结果：** 未缩放点积在这个随机例子中更容易产生接近 0 或明显偏大的权重。缩放不会让权重必然均匀，而是让不同特征维度下的相似度保持更可控的范围，使训练初期的 Softmax 和梯度不至于过早饱和。

## 3. Padding mask 与 causal mask

我们为四个时间步计算 Self-Attention。第一幅图把最后一列视为补齐位置，因此所有 Query 都不能读取；第二幅图使用 causal mask，因此第 $t$ 行只能读取不晚于自己的位置。

In [ ]:
sequence_tokens = np.array([[0.9, 0.1], [0.4, 0.6], [0.1, 1.0], [0.0, 0.0]], dtype=np.float64)  # 定义三个真实 token 和一个补齐 token
padding_mask = np.array([[0.0, 0.0, 0.0, -1e9]] * 4)  # 禁止全部 Query 读取最后补齐列
causal_mask = np.triu(np.full((4, 4), -1e9), k=1)  # 使用上三角大负数禁止读取未来位置
_, padding_weights, _ = scaled_dot_attention(sequence_tokens, sequence_tokens, sequence_tokens, padding_mask)  # 计算带 padding mask 的 Self-Attention
_, causal_weights, _ = scaled_dot_attention(sequence_tokens, sequence_tokens, sequence_tokens, causal_mask)  # 计算带 causal mask 的 Self-Attention
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))  # 创建两个注意力矩阵坐标轴
for axis, weights, title in zip(axes, [padding_weights, causal_weights], ["Padding mask", "Causal mask"]):  # 依次显示两种 mask 的权重矩阵
    image = axis.imshow(weights, vmin=0.0, vmax=1.0, cmap="Blues")  # 使用统一颜色范围绘制权重热图
    axis.set(title=title, xlabel="Key position", ylabel="Query position")  # 标注矩阵行列含义
    axis.set_xticks(range(4))  # 显示四个 Key 位置刻度
    axis.set_yticks(range(4))  # 显示四个 Query 位置刻度
    for row in range(4):  # 遍历权重矩阵的每一行
        for column in range(4):  # 遍历当前行的每一列
            axis.text(column, row, f"{weights[row, column]:.2f}", ha="center", va="center", color="black")  # 在热图中写出具体权重
fig.colorbar(image, ax=axes, shrink=0.82, label="Attention weight")  # 为两幅图添加共用颜色条
plt.show()  # 显示两种 mask 对信息流的影响

**怎样理解结果：** Padding mask 让最后一列始终为零，但真实 Query 之间仍可双向读取；causal mask 则形成下三角权重，第 0 行只能看自己，第 3 行可以读取此前全部位置。被 mask 的位置不是“权重较小”，而是被明确排除在 Softmax 归一化之外。

**本练习的结论：** Attention 的输出由 Query、Key、Value、缩放和 mask 共同决定。权重图适合检查信息是否按预期流动，但不能单独证明某个位置对最终机器人动作具有因果作用。